In [14]:
from dataclasses import dataclass
from collections import Counter
import random
import argparse
from typing import List, Dict, Tuple
import os
import csv
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm

%matplotlib inline



In [ ]:
#scoring system

@dataclass(frozen=True)
class ScoreSystem:
    """Pure numeric scoring: tens = max die, ones = min die."""
    name: str = "numeric"

    def score_value(self, d1: int, d2: int) -> int:
        a, b = max(d1, d2), min(d1, d2)
        return 10 * a + b

    def label(self, score_value: int) -> str:
        tens = score_value // 10
        ones = score_value % 10
        return f"{tens}{ones}"


def build_distribution(sys: ScoreSystem) -> Tuple[List[int], List[float], Dict[int, int]]:
    """
    PMF over distinct score values induced by 36 ordered rolls.
    Doubles: 1/36; non-doubles: 2/36.
    """
    counts = Counter()
    for d1 in range(1, 7):
        for d2 in range(1, 7):
            sv = sys.score_value(d1, d2)
            counts[sv] += 1
    values = sorted(counts.keys())  # worst -> best
    total = 36.0
    pmf = [counts[v] / total for v in values]
    index_of = {v: i for i, v in enumerate(values)}
    return values, pmf, index_of


def cdf_from_pmf(pmf: List[float]) -> List[float]:
    c, acc = [], 0.0
    for p in pmf:
        acc += p
        c.append(acc)
    return c


In [3]:
#exact survival

def survive_stop_exact(F: List[float], idx: int, rolls_allowed: int, opponents: int) -> float:
    """
    Survival if you STOP at score index 'idx' after 'rolls_allowed' total rolls,
    with 'opponents' opponents who may take up to the same number of rolls and
    stop as soon as they strictly beat your score.

    Eliminated iff ALL opponents strictly beat your score:
      Survival = 1 - (1 - F[s]^r)^K,  where F[s] = P(score <= s) on one roll,
      r = rolls_allowed, K = opponents.
    """
    F_s = F[idx]
    return 1.0 - (1.0 - (F_s ** rolls_allowed)) ** opponents


def expected_value_roll_forced(values: List[int], pmf: List[float],
                               F: List[float], opponents: int, rolls_allowed: int) -> float:
    """EV if you choose to roll (no more decisions)."""
    ev = 0.0
    for i, p in enumerate(pmf):
        ev += p * survive_stop_exact(F, i, rolls_allowed, opponents)
    return ev


def solve_thresholds_exact(sys: ScoreSystem, players: int) -> Dict[str, object]:
    """
    Optimal first-player thresholds (after 1st and after 2nd roll) via exact calculus.
    Also returns helper arrays for plotting.
    """
    opponents = players - 1
    values, pmf, _ = build_distribution(sys)
    F = cdf_from_pmf(pmf)

    # Second decision: stop now (r=2) vs take forced 3rd (r=3)
    ev_roll_third = expected_value_roll_forced(values, pmf, F, opponents, rolls_allowed=3)
    def stop2(i): return survive_stop_exact(F, i, rolls_allowed=2, opponents=opponents)
    idx2 = min(i for i in range(len(values)) if stop2(i) >= ev_roll_third)

    # First decision: stop now (r=1) vs continue (draw S2 then choose best of stop2 or roll3)
    ev_continue_first = 0.0
    for i, p in enumerate(pmf):
        ev_continue_first += p * max(stop2(i), ev_roll_third)

    def stop1(i): return survive_stop_exact(F, i, rolls_allowed=1, opponents=opponents)
    idx1 = min(i for i in range(len(values)) if stop1(i) >= ev_continue_first)

    return {
        "players": players,
        "threshold_after_1st": values[idx1],
        "threshold_after_2nd": values[idx2],
        "threshold_indices": {"after_1st": idx1, "after_2nd": idx2},
        "threshold_labels": {
            "after_1st": sys.label(values[idx1]),
            "after_2nd": sys.label(values[idx2]),
        },
        "threshold_quantiles": {
            "after_1st": F[idx1],
            "after_2nd": F[idx2],
        },
        "EV": {
            "roll_forced_3rd": ev_roll_third,
            "continue_from_1st": ev_continue_first,
            "always_stop_r1": expected_value_roll_forced(values, pmf, F, opponents, rolls_allowed=1),
            "always_stop_r2": expected_value_roll_forced(values, pmf, F, opponents, rolls_allowed=2),
            "always_roll_r3": ev_roll_third,  # alias
        },
        "values": values,
        "pmf": pmf,
        "cdf": F,
    }

In [4]:
#monte carlo validator 

def sample_one_roll(sys: ScoreSystem, rng: random.Random) -> int:
    d1, d2 = rng.randint(1, 6), rng.randint(1, 6)
    return sys.score_value(d1, d2)


def chaser_final_vs_cut(sys: ScoreSystem, cut_value: int, rolls_allowed: int, rng: random.Random) -> int:
    """
    Opponent policy: up to 'rolls_allowed' rolls; stop as soon as a roll strictly beats 'cut_value',
    else keep last roll.
    """
    last = None
    for _ in range(rolls_allowed):
        s = sample_one_roll(sys, rng)
        last = s
        if s > cut_value:
            return s
    return last


def resolve_tie_last_place(tie_count: int, rng: random.Random, tie_rule: str) -> bool:
    """
    Resolve a tie for last place including the first player.
    Return True if the first player SURVIVES the tie; False if eliminated.
    tie_rule ∈ {"lose_ties","win_ties","reroll"}.
    """
    if tie_rule == "lose_ties":
        return False
    if tie_rule == "win_ties":
        return True

    # Sudden-death rerolls among 'tie_count' players incl. first player.
    while True:
        # Any consistent sudden-death will do; keep it simple.
        rolls = [random.randint(1, 6) * 10 + random.randint(1, 6) for _ in range(tie_count)]
        min_roll = min(rolls)
        losers = [i for i, s in enumerate(rolls) if s == min_roll]
        if len(losers) == 1:
            return losers[0] != 0
        tie_count = len(losers)


def mc_survival_if_stop(sys: ScoreSystem, players: int, stop_value: int,
                        rolls_allowed: int, trials: int, seed: int,
                        tie_rule: str = "reroll") -> float:
    """Monte Carlo survival if first player stops at 'stop_value' with 'rolls_allowed' total rolls."""
    K = players - 1
    rng = random.Random(seed)
    survive = 0
    for _ in range(trials):
        opp_scores = [chaser_final_vs_cut(sys, stop_value, rolls_allowed, rng) for _ in range(K)]
        all_scores = [stop_value] + opp_scores
        m = min(all_scores)
        if m < stop_value:
            survive += 1
        elif m == stop_value:
            tie_count = sum(1 for s in all_scores if s == m)
            if resolve_tie_last_place(tie_count, rng, tie_rule):
                survive += 1
    return survive / trials


def mc_ev_roll_forced(sys: ScoreSystem, players: int, rolls_allowed: int,
                      trials: int, seed: int, tie_rule: str = "reroll") -> float:
    """Monte Carlo EV if you choose to roll (no decisions)."""
    rng = random.Random(seed)
    survive = 0
    for _ in range(trials):
        my_s = sample_one_roll(sys, rng)
        K = players - 1
        opp_scores = [chaser_final_vs_cut(sys, my_s, rolls_allowed, rng) for _ in range(K)]
        all_scores = [my_s] + opp_scores
        m = min(all_scores)
        if m < my_s:
            survive += 1
        elif m == my_s:
            tie_count = sum(1 for s in all_scores if s == m)
            if resolve_tie_last_place(tie_count, rng, tie_rule):
                survive += 1
    return survive / trials

In [19]:
def ensure_dir(path: str):
    if not os.path.isdir(path):
        os.makedirs(path, exist_ok=True)


def plot_opening_moves(sys: ScoreSystem, res: Dict[str, object],
                       save_dir: str, fmt: str = "png", dpi: int = 180):
    """
    Save a 2-panel figure for the opener's decisions:
      • Panel A: Stop after 1st (curve) vs Continue (flat).
      • Panel B: Stop after 2nd (curve) vs Roll 3rd (flat).
    """
    players = res["players"]
    K = players - 1
    values = res["values"]
    F = res["cdf"]
    idx1 = res["threshold_indices"]["after_1st"]
    idx2 = res["threshold_indices"]["after_2nd"]
    t1_label = res["threshold_labels"]["after_1st"]
    t2_label = res["threshold_labels"]["after_2nd"]
    ev1_cont = res["EV"]["continue_from_1st"]
    ev2_roll3 = res["EV"]["roll_forced_3rd"]

    s_stop_r1 = [survive_stop_exact(F, i, 1, K) for i in range(len(values))]
    s_stop_r2 = [survive_stop_exact(F, i, 2, K) for i in range(len(values))]

    x = list(range(len(values)))
    xticklabels = [sys.label(v) for v in values]

    fig, axes = plt.subplots(2, 1, figsize=(9.5, 8.0), constrained_layout=True)

    # Panel A
    ax = axes[0]
    ax.plot(x, s_stop_r1, lw=2, label="Stop after 1st (exact)")
    ax.axhline(ev1_cont, linestyle="--", linewidth=2, label="Continue after 1st (EV)")
    ax.axvline(idx1, linestyle="--", linewidth=1.5)
    ax.annotate(f"Threshold {t1_label}", xy=(idx1, s_stop_r1[idx1]),
                xytext=(idx1 + 0.5, min(0.98, s_stop_r1[idx1] + 0.05)),
                arrowprops=dict(arrowstyle="->", lw=1))
    ax.set_title(f"First decision (players={players})")
    ax.set_ylabel("Survival probability")
    ax.set_xticks(x)
    ax.set_xticklabels(xticklabels, rotation=0)
    ax.set_ylim(0, 1.0)
    ax.grid(True, alpha=0.25)
    ax.legend(loc="lower right")

    # Panel B
    ax = axes[1]
    ax.plot(x, s_stop_r2, lw=2, label="Stop after 2nd (exact)")
    ax.axhline(ev2_roll3, linestyle="--", linewidth=2, label="Roll 3rd (EV)")
    ax.axvline(idx2, linestyle="--", linewidth=1.5)
    ax.annotate(f"Threshold {t2_label}", xy=(idx2, s_stop_r2[idx2]),
                xytext=(idx2 + 0.5, min(0.98, s_stop_r2[idx2] + 0.05)),
                arrowprops=dict(arrowstyle="->", lw=1))
    ax.set_title(f"Second decision (players={players})")
    ax.set_xlabel("Your score (higher is better)")
    ax.set_ylabel("Survival probability")
    ax.set_xticks(x)
    ax.set_xticklabels(xticklabels, rotation=0)
    ax.set_ylim(0, 1.0)
    ax.grid(True, alpha=0.25)
    ax.legend(loc="lower right")

    ensure_dir(save_dir)
    base = os.path.join(save_dir, f"mexico_numeric_opening_p{players}")
    fig.savefig(base + f".{fmt}", dpi=dpi)
    if fmt.lower() != "pdf":
        fig.savefig(base + ".pdf", dpi=300)
    plt.show()

plt.show() 

In [15]:
def plot_opening_moves(sys, res):
    """
    Display a 2-panel figure for the opener's decisions:
      • Panel A: Stop after 1st (curve) vs Continue (flat).
      • Panel B: Stop after 2nd (curve) vs Roll 3rd (flat).
    """
    players = res["players"]
    K = players - 1
    values = res["values"]
    F = res["cdf"]
    idx1 = res["threshold_indices"]["after_1st"]
    idx2 = res["threshold_indices"]["after_2nd"]
    t1_label = res["threshold_labels"]["after_1st"]
    t2_label = res["threshold_labels"]["after_2nd"]
    ev1_cont = res["EV"]["continue_from_1st"]
    ev2_roll3 = res["EV"]["roll_forced_3rd"]

    # Compute survival probabilities
    s_stop_r1 = [survive_stop_exact(F, i, 1, K) for i in range(len(values))]
    s_stop_r2 = [survive_stop_exact(F, i, 2, K) for i in range(len(values))]

    x = list(range(len(values)))
    xticklabels = [sys.label(v) for v in values]

    # Create figure
    fig, axes = plt.subplots(2, 1, figsize=(9.5, 8.0), constrained_layout=True)

    # Panel A
    ax = axes[0]
    ax.plot(x, s_stop_r1, lw=2, label="Stop after 1st (exact)")
    ax.axhline(ev1_cont, linestyle="--", linewidth=2, label="Continue after 1st (EV)")
    ax.axvline(idx1, linestyle="--", linewidth=1.5)
    ax.annotate(f"Threshold {t1_label}", xy=(idx1, s_stop_r1[idx1]),
                xytext=(idx1 + 0.5, min(0.98, s_stop_r1[idx1] + 0.05)),
                arrowprops=dict(arrowstyle="->", lw=1))
    ax.set_title(f"First decision (players={players})")
    ax.set_ylabel("Survival probability")
    ax.set_xticks(x)
    ax.set_xticklabels(xticklabels, rotation=0)
    ax.set_ylim(0, 1.0)
    ax.grid(True, alpha=0.25)
    ax.legend(loc="lower right")

    # Panel B
    ax = axes[1]
    ax.plot(x, s_stop_r2, lw=2, label="Stop after 2nd (exact)")
    ax.axhline(ev2_roll3, linestyle="--", linewidth=2, label="Roll 3rd (EV)")
    ax.axvline(idx2, linestyle="--", linewidth=1.5)
    ax.annotate(f"Threshold {t2_label}", xy=(idx2, s_stop_r2[idx2]),
                xytext=(idx2 + 0.5, min(0.98, s_stop_r2[idx2] + 0.05)),
                arrowprops=dict(arrowstyle="->", lw=1))
    ax.set_title(f"Second decision (players={players})")
    ax.set_xlabel("Your score (higher is better)")
    ax.set_ylabel("Survival probability")
    ax.set_xticks(x)
    ax.set_xticklabels(xticklabels, rotation=0)
    ax.set_ylim(0, 1.0)
    ax.grid(True, alpha=0.25)
    ax.legend(loc="lower right")

    # Display figure inline
    plt.show()

In [ ]:
def gather_summary(sys: ScoreSystem, players_list: List[int]) -> Dict[str, List]:
    values, pmf, _ = build_distribution(sys)
    score_labels = [sys.label(v) for v in values]

    summary = {
        "players": [],
        "t1_value": [],
        "t2_value": [],
        "t1_label": [],
        "t2_label": [],
        "t1_quantile": [],
        "t2_quantile": [],
        "t1_idx": [],
        "t2_idx": [],
        "surv_t1": [],
        "surv_t2": [],
        "ev_optimal": [],
        "ev_stop1": [],
        "ev_stop2": [],
        "ev_roll3": [],
        "stop_curves_r1": [],  # list of lists
        "stop_curves_r2": [],  # list of lists
        "adv_heat_r1": [],     # stop_r1 - continueEV (per score)
        "adv_heat_r2": [],     # stop_r2 - roll3EV (per score)
        "score_labels": score_labels,
        "values": values,
    }

    for p in players_list:
        res = solve_thresholds_exact(sys, p)
        K = p - 1
        F = res["cdf"]
        idx1 = res["threshold_indices"]["after_1st"]
        idx2 = res["threshold_indices"]["after_2nd"]

        s_stop_r1 = [survive_stop_exact(F, i, 1, K) for i in range(len(values))]
        s_stop_r2 = [survive_stop_exact(F, i, 2, K) for i in range(len(values))]

        contEV = res["EV"]["continue_from_1st"]
        roll3EV = res["EV"]["roll_forced_3rd"]

        adv1 = [s - contEV for s in s_stop_r1]
        adv2 = [s - roll3EV for s in s_stop_r2]

        summary["players"].append(p)
        summary["t1_value"].append(res["threshold_after_1st"])
        summary["t2_value"].append(res["threshold_after_2nd"])
        summary["t1_label"].append(res["threshold_labels"]["after_1st"])
        summary["t2_label"].append(res["threshold_labels"]["after_2nd"])
        summary["t1_quantile"].append(res["threshold_quantiles"]["after_1st"])
        summary["t2_quantile"].append(res["threshold_quantiles"]["after_2nd"])
        summary["t1_idx"].append(idx1)
        summary["t2_idx"].append(idx2)
        summary["surv_t1"].append(s_stop_r1[idx1])
        summary["surv_t2"].append(s_stop_r2[idx2])
        summary["ev_optimal"].append(contEV)
        summary["ev_stop1"].append(res["EV"]["always_stop_r1"])
        summary["ev_stop2"].append(res["EV"]["always_stop_r2"])
        summary["ev_roll3"].append(res["EV"]["roll_forced_3rd"])
        summary["stop_curves_r1"].append(s_stop_r1)
        summary["stop_curves_r2"].append(s_stop_r2)
        summary["adv_heat_r1"].append(adv1)
        summary["adv_heat_r2"].append(adv2)

    return summary


def plot_thresholds_vs_players(sys: ScoreSystem, summary: Dict[str, List], save_dir: str, fmt: str, dpi: int):
    plt.figure(figsize=(8.5, 5.0))
    plt.plot(summary["players"], summary["t1_value"], lw=2, label="Threshold after 1st (score)")
    plt.plot(summary["players"], summary["t2_value"], lw=2, label="Threshold after 2nd (score)")
    plt.xlabel("Players at table")
    plt.ylabel("Threshold score (two-digit)")
    plt.title("Optimal opener thresholds vs number of players")
    plt.grid(True, alpha=0.25)
    plt.legend(loc="best")
    ensure_dir(save_dir)
    base = os.path.join(save_dir, "summary_thresholds_vs_players")
    plt.savefig(base + f".{fmt}", dpi=dpi)
    if fmt.lower() != "pdf":
        plt.savefig(base + ".pdf", dpi=300)
    plt.show()


def plot_threshold_quantiles_vs_players(summary: Dict[str, List], save_dir: str, fmt: str, dpi: int):
    plt.figure(figsize=(8.5, 5.0))
    plt.plot(summary["players"], summary["t1_quantile"], lw=2, label="Quantile of 1st threshold (F)")
    plt.plot(summary["players"], summary["t2_quantile"], lw=2, label="Quantile of 2nd threshold (F)")
    plt.xlabel("Players at table")
    plt.ylabel("One-roll CDF F(threshold)")
    plt.title("Threshold quantiles vs number of players")
    plt.ylim(0, 1.0)
    plt.grid(True, alpha=0.25)
    plt.legend(loc="best")
    ensure_dir(save_dir)
    base = os.path.join(save_dir, "summary_threshold_quantiles_vs_players")
    plt.savefig(base + f".{fmt}", dpi=dpi)
    if fmt.lower() != "pdf":
        plt.savefig(base + ".pdf", dpi=300)


def plot_survival_at_thresholds_vs_players(summary: Dict[str, List], save_dir: str, fmt: str, dpi: int):
    plt.figure(figsize=(8.5, 5.0))
    plt.plot(summary["players"], summary["surv_t1"], lw=2, label="Survival at 1st threshold")
    plt.plot(summary["players"], summary["surv_t2"], lw=2, label="Survival at 2nd threshold")
    plt.xlabel("Players at table")
    plt.ylabel("Survival probability")
    plt.title("Survival at optimal thresholds vs number of players")
    plt.ylim(0, 1.0)
    plt.grid(True, alpha=0.25)
    plt.legend(loc="best")
    ensure_dir(save_dir)
    base = os.path.join(save_dir, "summary_survival_at_thresholds_vs_players")
    plt.savefig(base + f".{fmt}", dpi=dpi)
    if fmt.lower() != "pdf":
        plt.savefig(base + ".pdf", dpi=300)


def plot_policy_ev_vs_players(summary: Dict[str, List], save_dir: str, fmt: str, dpi: int):
    plt.figure(figsize=(8.5, 5.0))
    plt.plot(summary["players"], summary["ev_optimal"], lw=2, label="Optimal policy")
    plt.plot(summary["players"], summary["ev_stop1"], lw=2, label="Always stop after 1st")
    plt.plot(summary["players"], summary["ev_stop2"], lw=2, label="Always stop after 2nd")
    plt.plot(summary["players"], summary["ev_roll3"], lw=2, label="Always roll 3")
    plt.xlabel("Players at table")
    plt.ylabel("Survival probability (EV)")
    plt.title("Policy EV vs number of players (opener)")
    plt.ylim(0, 1.0)
    plt.grid(True, alpha=0.25)
    plt.legend(loc="best")
    ensure_dir(save_dir)
    base = os.path.join(save_dir, "summary_policy_ev_vs_players")
    plt.savefig(base + f".{fmt}", dpi=dpi)
    if fmt.lower() != "pdf":
        plt.savefig(base + ".pdf", dpi=300)


def plot_stop_curve_overlays(sys: ScoreSystem, summary: Dict[str, List], save_dir: str, fmt: str, dpi: int):
    # r = 1 overlay
    plt.figure(figsize=(9.5, 6.0))
    x = list(range(len(summary["score_labels"])))
    xticklabels = summary["score_labels"]
    for p, curve in zip(summary["players"], summary["stop_curves_r1"]):
        plt.plot(x, curve, lw=1.75, label=f"p={p}")
    plt.xlabel("Score (worst → best)")
    plt.ylabel("Survival if stop after 1st")
    plt.title("Stop-after-1st curves over number of players")
    plt.xticks(x, xticklabels, rotation=0)
    plt.ylim(0, 1.0)
    plt.grid(True, alpha=0.25)
    plt.legend(loc="lower right", ncol=2)
    ensure_dir(save_dir)
    base = os.path.join(save_dir, "overlay_stop_curves_r1")
    plt.savefig(base + f".{fmt}", dpi=dpi)
    if fmt.lower() != "pdf":
        plt.savefig(base + ".pdf", dpi=300)

    # r = 2 overlay
    plt.figure(figsize=(9.5, 6.0))
    for p, curve in zip(summary["players"], summary["stop_curves_r2"]):
        plt.plot(x, curve, lw=1.75, label=f"p={p}")
    plt.xlabel("Score (worst → best)")
    plt.ylabel("Survival if stop after 2nd")
    plt.title("Stop-after-2nd curves over number of players")
    plt.xticks(x, xticklabels, rotation=0)
    plt.ylim(0, 1.0)
    plt.grid(True, alpha=0.25)
    plt.legend(loc="lower right", ncol=2)
    ensure_dir(save_dir)
    base = os.path.join(save_dir, "overlay_stop_curves_r2")
    plt.savefig(base + f".{fmt}", dpi=dpi)
    if fmt.lower() != "pdf":
        plt.savefig(base + ".pdf", dpi=300)


def plot_advantage_heatmaps(summary: Dict[str, List], save_dir: str, fmt: str, dpi: int):
    """
    Zero-centered, comparable heatmaps with shared scale and threshold overlays.
    r1 shows Stop − ContinueEV; r2 shows Stop − Roll3EV.
    """
    ensure_dir(save_dir)
    players = summary["players"]
    scores  = summary["score_labels"]

    Z1 = np.array(summary["adv_heat_r1"])  # shape: [num_players, num_scores]
    Z2 = np.array(summary["adv_heat_r2"])

    # Shared symmetric scale across both maps
    M = float(np.max(np.abs([Z1, Z2])))
    M = max(M, 1e-6)  # guard against all-zeros
    norm = TwoSlopeNorm(vmin=-M, vcenter=0.0, vmax=M)
    ticks = np.round(np.linspace(-M, M, 5), 2)

    def _one(Z, tag, title, cbar_label, t_indices):
        fig, ax = plt.subplots(figsize=(10.5, 5.8))
        im = ax.imshow(Z, aspect="auto", origin="lower", cmap="coolwarm", norm=norm)
        # 0-contour
        ax.contour(Z, levels=[0.0], colors="k", linewidths=1.0, origin="lower")

        # Overlay threshold markers: (row i, col t_indices[i])
        yy = np.arange(len(players))
        xx = np.array(t_indices, dtype=float)
        ax.plot(xx, yy, "w.", ms=6, label="threshold")

        ax.set_xticks(np.arange(len(scores)))
        ax.set_xticklabels(scores, rotation=0)
        ax.set_yticks(np.arange(len(players)))
        ax.set_yticklabels(players)
        ax.set_xlabel("Score (worst → best)")
        ax.set_ylabel("Players at table")
        ax.set_title(title)

        cbar = fig.colorbar(im, ax=ax, pad=0.02)
        cbar.set_label(cbar_label)
        cbar.set_ticks(ticks)

        ax.legend(loc="upper left", frameon=True)
        fig.tight_layout()

        base = os.path.join(save_dir, f"heatmap_advantage_{tag}_centered")
        fig.savefig(base + f".{fmt}", dpi=dpi)
        if fmt.lower() != "pdf":
            fig.savefig(base + ".pdf", dpi=300)

    _one(Z1, "r1",
         "Decision advantage after 1st (Stop − ContinueEV)",
         "Advantage (probability points)", summary["t1_idx"])

    _one(Z2, "r2",
         "Decision advantage after 2nd (Stop − Roll3EV)",
         "Advantage (probability points)", summary["t2_idx"])


def plot_decision_maps(summary: Dict[str, List], save_dir: str, fmt: str, dpi: int):
    """
    Discrete maps: 1 = Stop better, 0 = Continue/Roll better.
    Overlays threshold markers.
    """
    ensure_dir(save_dir)
    players = summary["players"]
    scores  = summary["score_labels"]

    Z1 = (np.array(summary["adv_heat_r1"]) >= 0).astype(int)
    Z2 = (np.array(summary["adv_heat_r2"]) >= 0).astype(int)

    def _one(Z, tag, title, t_indices):
        fig, ax = plt.subplots(figsize=(10.5, 5.0))
        im = ax.imshow(Z, aspect="auto", origin="lower", cmap="bwr", vmin=0, vmax=1)
        yy = np.arange(len(players))
        xx = np.array(t_indices, dtype=float)
        ax.plot(xx, yy, "k.", ms=6, label="threshold")

        ax.set_xticks(np.arange(len(scores)))
        ax.set_xticklabels(scores, rotation=0)
        ax.set_yticks(np.arange(len(players)))
        ax.set_yticklabels(players)
        ax.set_xlabel("Score (worst → best)")
        ax.set_ylabel("Players at table")
        ax.set_title(title)

        cbar = fig.colorbar(im, ax=ax, pad=0.02)
        cbar.set_ticks([0, 1])
        cbar.set_ticklabels(["Continue/Roll better", "Stop better"])
        ax.legend(loc="upper left")
        fig.tight_layout()

        base = os.path.join(save_dir, f"decision_map_{tag}")
        fig.savefig(base + f".{fmt}", dpi=dpi)
        if fmt.lower() != "pdf":
            fig.savefig(base + ".pdf", dpi=300)

    _one(Z1, "r1", "Decision map after 1st (Stop vs Continue)", summary["t1_idx"])
    _one(Z2, "r2", "Decision map after 2nd (Stop vs Roll 3rd)", summary["t2_idx"])








In [ ]:
def plot_tie_rule_sensitivity(sys: ScoreSystem, summary: Dict[str, List], save_dir: str,
                              fmt: str, dpi: int, mc_trials: int, seed: int):
    """
    For each players p, show MC survival at the thresholds under three tie rules.
    We do both thresholds (after 1st and after 2nd) in two subplots.
    """
    if mc_trials <= 0:
        return
    ensure_dir(save_dir)
    players = summary["players"]

    # First threshold
    vals_reroll, vals_lose, vals_win = [], [], []
    for i, p in enumerate(players):
        t = summary["t1_value"][i]
        s_reroll = mc_survival_if_stop(ScoreSystem(), p, t, 1, mc_trials, seed + i*3 + 0, "reroll")
        s_lose   = mc_survival_if_stop(ScoreSystem(), p, t, 1, mc_trials, seed + i*3 + 1, "lose_ties")
        s_win    = mc_survival_if_stop(ScoreSystem(), p, t, 1, mc_trials, seed + i*3 + 2, "win_ties")
        vals_reroll.append(s_reroll)
        vals_lose.append(s_lose)
        vals_win.append(s_win)

    # Second threshold
    vals_reroll2, vals_lose2, vals_win2 = [], [], []
    for i, p in enumerate(players):
        t = summary["t2_value"][i]
        s_reroll = mc_survival_if_stop(ScoreSystem(), p, t, 2, mc_trials, seed + 1000 + i*3 + 0, "reroll")
        s_lose   = mc_survival_if_stop(ScoreSystem(), p, t, 2, mc_trials, seed + 1000 + i*3 + 1, "lose_ties")
        s_win    = mc_survival_if_stop(ScoreSystem(), p, t, 2, mc_trials, seed + 1000 + i*3 + 2, "win_ties")
        vals_reroll2.append(s_reroll)
        vals_lose2.append(s_lose)
        vals_win2.append(s_win)

    # Plot side-by-side bars
    fig, axes = plt.subplots(2, 1, figsize=(10.5, 8.0), constrained_layout=True)

    # Top: after 1st threshold
    ax = axes[0]
    xs = range(len(players))
    width = 0.25
    ax.bar([x - width for x in xs], vals_reroll, width, label="reroll")
    ax.bar(xs, vals_lose, width, label="lose_ties")
    ax.bar([x + width for x in xs], vals_win, width, label="win_ties")
    ax.set_xticks(list(xs))
    ax.set_xticklabels(players)
    ax.set_ylim(0, 1.0)
    ax.set_ylabel("Survival at 1st threshold")
    ax.set_title("Tie-rule sensitivity (Monte Carlo)")
    ax.grid(True, axis="y", alpha=0.25)
    ax.legend(loc="best")

    # Bottom: after 2nd threshold
    ax = axes[1]
    ax.bar([x - width for x in xs], vals_reroll2, width, label="reroll")
    ax.bar(list(xs), vals_lose2, width, label="lose_ties")
    ax.bar([x + width for x in xs], vals_win2, width, label="win_ties")
    ax.set_xticks(list(xs))
    ax.set_xticklabels(players)
    ax.set_ylim(0, 1.0)
    ax.set_xlabel("Players at table")
    ax.set_ylabel("Survival at 2nd threshold")
    ax.grid(True, axis="y", alpha=0.25)
    ax.legend(loc="best")

    base = os.path.join(save_dir, "tie_rule_sensitivity")
    fig.savefig(base + f".{fmt}", dpi=dpi)
    if fmt.lower() != "pdf":
        fig.savefig(base + ".pdf", dpi=300)